# CNN-GMM — Day 1: Setup & Verification

**Goal:** Confirm the shared data pipeline produces results consistent with `gp.ipynb`.

This notebook should be run once and kept as a permanent record.  
Every number printed here must match the corresponding output in `gp.ipynb`.

| Check | Expected (from gp.ipynb) | Actual |
|-------|--------------------------|--------|
| Total samples | ~85,000 | ??? |
| GC count | ~18,500 | ??? |
| non-GC count | ~67,000 | ??? |
| Train size | ~59,500 | ??? |
| Val size | ~8,500 | ??? |
| Test size | ~17,000 | ??? |
| Frame shape | (2, 20, 20) | ??? |
| CLASS_WEIGHTS[0] (non-GC) | < 1.0 | ??? |
| CLASS_WEIGHTS[1] (GC) | > 1.0 | ??? |

## 0 · Seeds — Set First, Before Anything Else

In [ ]:
# ── This cell must be the first executable cell in every notebook. ──────────
# Seeds are set BEFORE imports that might trigger random operations.

import sys
sys.path.insert(0, '..')   # adjust if your repo layout differs

from utils.shared_utils import set_seeds
set_seeds(42)   # ← DO NOT change this value

## 1 · Imports

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from shared_utils import (
    load_all_frames,
    compute_class_weights,
    make_splits,
    make_loader,
    FrameDataset,
    verify_setup,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2 · Load Data

In [ ]:
all_frames, fcc_df, vcc_df = load_all_frames()

print(f'all_frames shape : {all_frames.shape}')
print(f'fcc_df shape     : {fcc_df.shape}')
print(f'vcc_df shape     : {vcc_df.shape}')
print()
print('Columns:', list(all_frames.columns))
print()
print('First frame dtype :', all_frames.frame.iloc[0].dtype)
print('First frame shape :', all_frames.frame.iloc[0].shape)

## 3 · Class Weights

These **must** match `gp.ipynb`. The formula is:
```
w_pos = total / (2 * n_pos)
w_neg = total / (2 * n_neg)
CLASS_WEIGHTS = [w_neg, w_pos]   # index 0 = non-GC, index 1 = GC
```

In [ ]:
CLASS_WEIGHTS, stats = compute_class_weights(all_frames, device)

print('--- Class statistics ---')
print(f"Total samples : {stats['total']:,}")
print(f"  GC  (pos=1) : {stats['n_pos']:,}  ({100*stats['n_pos']/stats['total']:.1f} %)")
print(f"  non-GC (0)  : {stats['n_neg']:,}  ({100*stats['n_neg']/stats['total']:.1f} %)")
print()
print('--- Class weights ---')
print(f"  CLASS_WEIGHTS[0] (non-GC) : {CLASS_WEIGHTS[0].item():.4f}")
print(f"  CLASS_WEIGHTS[1] (GC)     : {CLASS_WEIGHTS[1].item():.4f}")
print(f"  Ratio GC/non-GC weight    : {CLASS_WEIGHTS[1].item()/CLASS_WEIGHTS[0].item():.2f}x")
print()
print('>>> Compare these numbers with gp.ipynb output. They must match. <<<')

## 4 · Train / Val / Test Split

Split is 70 / 10 / 20 with `seed=42` — identical to `gp.ipynb`.

In [ ]:
train_df, val_df, test_df = make_splits(all_frames, seed=42)

print(f'Train : {len(train_df):,}  ({100*len(train_df)/stats["total"]:.1f} %)')
print(f'Val   : {len(val_df):,}  ({100*len(val_df)/stats["total"]:.1f} %)')
print(f'Test  : {len(test_df):,}  ({100*len(test_df)/stats["total"]:.1f} %)')
print()
print('GC fraction in each split:')
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    gc_frac = df['y'].mean()
    print(f'  {name:5s} : {gc_frac:.3f}  (should be ~{stats["n_pos"]/stats["total"]:.3f})')

## 5 · DataLoaders — Batch Shape Check

In [ ]:
train_dl = make_loader(train_df, batch_size=64, shuffle=True,  num_workers=4)
val_dl   = make_loader(val_df,   batch_size=64, shuffle=False, num_workers=4)
test_dl  = make_loader(test_df,  batch_size=64, shuffle=False, num_workers=4)

# Grab one batch and inspect it
x_batch, y_batch = next(iter(train_dl))

print('--- First training batch ---')
print(f'  x_batch.shape : {x_batch.shape}   (expected: torch.Size([64, 2, 20, 20]))')
print(f'  x_batch.dtype : {x_batch.dtype}   (expected: torch.float32)')
print(f'  y_batch.shape : {y_batch.shape}   (expected: torch.Size([64]))')
print(f'  y_batch.dtype : {y_batch.dtype}   (expected: torch.int64)')
print(f'  y_batch unique values: {y_batch.unique().tolist()}  (expected: [0, 1])')
print()
print(f'  x pixel range : [{x_batch.min():.3f}, {x_batch.max():.3f}]')
print(f'  x mean/std    : {x_batch.mean():.4f} / {x_batch.std():.4f}')

## 6 · Visual Sanity Check — Sample Images

In [ ]:
# Show 8 GC and 8 non-GC examples side by side.
# Each source has two channels (g=top, z=bottom).

gc_df     = train_df[train_df['y'] == True].head(8)
non_gc_df = train_df[train_df['y'] == False].head(8)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle('Sample sources — left 8: GC  |  right 8: non-GC', fontsize=13)

for col, (_, row) in enumerate(gc_df.iterrows()):
    frame = row['frame']                       # (2, 20, 20)
    axes[0, col].imshow(frame[0], cmap='gray', origin='lower')
    axes[1, col].imshow(frame[1], cmap='gray', origin='lower')
    axes[0, col].set_title('GC g', fontsize=7)
    axes[1, col].set_title('GC z', fontsize=7)
    for ax in axes[:2, col]: ax.axis('off')

for col, (_, row) in enumerate(non_gc_df.iterrows()):
    frame = row['frame']
    axes[2, col].imshow(frame[0], cmap='gray', origin='lower')
    axes[3, col].imshow(frame[1], cmap='gray', origin='lower')
    axes[2, col].set_title('non-GC g', fontsize=7)
    axes[3, col].set_title('non-GC z', fontsize=7)
    for ax in axes[2:, col]: ax.axis('off')

plt.tight_layout()
plt.savefig('day1_sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: day1_sample_images.png')

## 7 · Class Distribution Plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
sns.histplot(data=all_frames, x='y', bins=2, hue='y',
             palette=['steelblue', 'tomato'], ax=ax)
ax.set_title('Full dataset — class distribution')
ax.set_xticks([0, 1])
ax.set_xticklabels(['non-GC (0)', 'GC (1)'])
plt.tight_layout()
plt.savefig('day1_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: day1_class_distribution.png')

## 8 · Cross-Cluster Split Sizes

Used later in the transfer experiments (train VCC → test FCC and vice versa).

In [ ]:
print('--- Cross-cluster dataset sizes ---')
print(f'FCC (Fornax) : {len(fcc_df):,} sources')
print(f'  GC     : {fcc_df["y"].sum():,}  ({100*fcc_df["y"].mean():.1f} %)')
print(f'  non-GC : {(~fcc_df["y"]).sum():,}  ({100*(1-fcc_df["y"].mean()):.1f} %)')
print()
print(f'VCC (Virgo)  : {len(vcc_df):,} sources')
print(f'  GC     : {vcc_df["y"].sum():,}  ({100*vcc_df["y"].mean():.1f} %)')
print(f'  non-GC : {(~vcc_df["y"]).sum():,}  ({100*(1-vcc_df["y"].mean()):.1f} %)')

## 9 · Full Verification Suite

In [ ]:
verify_setup(
    all_frames    = all_frames,
    train_df      = train_df,
    val_df        = val_df,
    test_df       = test_df,
    class_weights = CLASS_WEIGHTS,
    stats         = stats,
)

## 10 · Record Numbers for Team Alignment

Fill in this table after running the notebook and share with your colleagues.
They should get the **same numbers** from their notebooks.

In [ ]:
# Auto-generate the alignment table
x0, y0 = next(iter(train_dl))

alignment = {
    'total_samples'         : stats['total'],
    'n_gc'                  : stats['n_pos'],
    'n_non_gc'              : stats['n_neg'],
    'train_size'            : len(train_df),
    'val_size'              : len(val_df),
    'test_size'             : len(test_df),
    'fcc_size'              : len(fcc_df),
    'vcc_size'              : len(vcc_df),
    'frame_shape'           : str(all_frames.frame.iloc[0].shape),
    'class_weight_non_gc'   : round(CLASS_WEIGHTS[0].item(), 6),
    'class_weight_gc'       : round(CLASS_WEIGHTS[1].item(), 6),
    'batch_x_shape'         : str(tuple(x0.shape)),
    'batch_y_shape'         : str(tuple(y0.shape)),
    'seed'                  : 42,
}

print('\n=== TEAM ALIGNMENT TABLE ===')
print('Share this with colleagues — all three notebooks must produce identical values.\n')
for k, v in alignment.items():
    print(f'  {k:<28} : {v}')

# Save as CSV for easy sharing
import json
with open('day1_alignment.json', 'w') as f:
    json.dump(alignment, f, indent=2)
print('\nSaved: day1_alignment.json')

---
## Day 1 Summary

If all cells above ran without assertion errors and the numbers match `gp.ipynb`:

- ✅ Seeds are consistent
- ✅ Data loads correctly with the right shape
- ✅ Train/val/test split is identical to the DCGP notebook
- ✅ Class weights use the exact same formula
- ✅ DataLoader produces correctly shaped batches

**Next step (Day 2):** Build the CNN baseline in `cnn_baseline.py`.